# Denoising Diffusion from Scratch — MNIST

**Practice project.** By the end of this notebook you will have written a complete
DDPM (Ho et al., 2020) — noise schedule, forward process, U-Net noise predictor,
training loop, and ancestral sampler — and used it to generate handwritten digits.

Plain PyTorch throughout. No `diffusers`, no pretrained weights.

### The one idea

A diffusion model is trained to answer a single question:

> *Given a noisy image and how much noise is in it, what was the noise?*

Repeatedly subtracting a little bit of predicted noise turns pure static into a digit.

### Roadmap

| Part | You implement | Why it matters |
|---|---|---|
| 0–1 | Setup and data | Getting MNIST into $[-1, 1]$ |
| 2 | Noise schedule ($\beta_t$, $\alpha_t$, $\bar\alpha_t$) | Defines *how fast* information is destroyed |
| 3 | Forward process $q(x_t \mid x_0)$ | The closed form that makes training cheap |
| 4 | U-Net with timestep conditioning | The network that predicts the noise |
| 5 | Training objective | Why the loss is just an MSE |
| 6 | Training loop | Putting it together |
| 7 | Ancestral sampler $p(x_{t-1} \mid x_t)$ | Turning noise into images |
| 8 | Evaluation — is it memorising? | Judging a generative model |
| 9–10 | *Bonus:* DDIM, class conditioning, CFG | The tricks real systems use |

### How to work through this

- Each part has a **`TODO`** you fill in, followed by a **✅ sanity-check cell** that must
  pass before you move on. If a check fails, the bug is in the cell right above it.
- Read the math in the markdown cells — each formula maps to one or two lines of code.
- Budget ~3–5 hours, plus training time.
- Get stuck for 20 minutes before opening the answer key.

### Requirements

```
pip install torch torchvision matplotlib tqdm
```

A GPU makes training ~10x faster, but this is CPU-survivable: set `train_subset = 8000`,
`epochs = 3`, and `timesteps = 400` in the config below.

> **This is the skeleton.** Search for `TODO`: 12 cells, in order, each 1–7 lines of code.
> Run the ✅ sanity-check cell under each part before moving on — a failing check means the
> bug is in the cell directly above it.
>
> Worked solutions live in `diffusion_mnist_solution.ipynb`. Try each TODO for 20 minutes
> before you open it — the sanity checks are designed to tell you *what* is wrong without
> telling you the answer.

## Part 0 — Setup

In [ ]:
import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = get_device()
print("torch", torch.__version__, "| device:", device)

In [ ]:
@dataclass
class Config:
    # data
    image_size: int = 28
    channels: int = 1
    batch_size: int = 128
    train_subset: int | None = None   # e.g. 8000 to iterate faster on CPU; None = all 60k

    # diffusion
    timesteps: int = 1000             # T
    schedule: str = "cosine"          # "cosine" or "linear"

    # model
    base_channels: int = 64
    channel_mults: tuple = (1, 2, 2)  # resolutions 28 -> 14 -> 7
    num_res_blocks: int = 2
    attn_resolutions: tuple = (7,)    # self-attention at 7x7 only
    dropout: float = 0.1

    # optimisation
    lr: float = 2e-4
    epochs: int = 8
    ema_decay: float = 0.999
    grad_clip: float = 1.0

    seed: int = 0


cfg = Config()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
cfg

## Part 1 — Data

MNIST pixels arrive in $[0, 1]$. We rescale to $[-1, 1]$ so the data roughly matches the
$\mathcal{N}(0, I)$ prior the sampler starts from at $t = T$. A mismatch here is one of the
most common silent bugs in diffusion code — the model trains fine and the samples look washed out.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                  # [0, 1], shape (1, 28, 28)
    transforms.Normalize((0.5,), (0.5,)),   # -> [-1, 1]
])

train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
if cfg.train_subset is not None:
    train_set = Subset(train_set, range(cfg.train_subset))

train_loader = DataLoader(
    train_set,
    batch_size=cfg.batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=0,      # num_workers > 0 can hang inside notebooks on Windows
)

x_batch, y_batch = next(iter(train_loader))
print("batch:", tuple(x_batch.shape), "| range:", (x_batch.min().item(), x_batch.max().item()))
print("labels:", y_batch[:8].tolist())

In [ ]:
def show(tensor, title=None, nrow=8, figsize=(8, 8)):
    """Display a batch of images that live in [-1, 1]."""
    grid = make_grid(tensor.detach().cpu().float().clamp(-1, 1), nrow=nrow, padding=2)
    grid = (grid + 1) / 2                     # back to [0, 1] for display
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray", vmin=0, vmax=1)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


show(x_batch[:32], title="Real MNIST samples", nrow=8, figsize=(8, 4))

## Part 2 — The noise schedule

The forward process adds a *little* Gaussian noise at each of $T$ steps:

$$q(x_t \mid x_{t-1}) = \mathcal{N}\!\left(x_t;\ \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I\right)$$

The schedule $\beta_1, \dots, \beta_T$ controls how fast the image is destroyed. Define

$$\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s .$$

$\bar{\alpha}_t$ is the quantity that actually matters: it runs from $\approx 1$ (clean image)
to $\approx 0$ (pure noise), and it is the only thing the forward process needs.

**Two schedules:**

- **Linear** (original DDPM): $\beta_t$ linearly spaced in $[10^{-4}, 0.02]$. Tuned for
  256×256 images — on 28×28 MNIST it destroys the image far too early, wasting most of
  the timesteps on indistinguishable static.
- **Cosine** (Nichol & Dhariwal, 2021): sets
  $\bar{\alpha}_t = f(t)/f(0)$ with $f(t) = \cos^2\!\left(\frac{t/T + s}{1 + s}\cdot\frac{\pi}{2}\right)$,
  then recovers $\beta_t = 1 - \bar{\alpha}_t / \bar{\alpha}_{t-1}$. Much better for small
  images — it is our default.

> **TODO:** implement `linear_beta_schedule`. The cosine one is given so you can compare them.

In [ ]:
def linear_beta_schedule(timesteps: int, beta_start: float = 1e-4, beta_end: float = 0.02):
    """Original DDPM schedule: betas linearly spaced from beta_start to beta_end.

    Returns a 1-D tensor of shape (timesteps,).
    """
    # TODO: one line. Hint: torch.linspace(..., dtype=torch.float64)
    raise NotImplementedError


def cosine_beta_schedule(timesteps: int, s: float = 0.008):
    """Nichol & Dhariwal (2021) cosine schedule. Provided for you."""
    steps = timesteps + 1
    t = torch.linspace(0, timesteps, steps, dtype=torch.float64) / timesteps
    alphas_cumprod = torch.cos((t + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return betas.clamp(1e-8, 0.999)


def make_betas(schedule: str, timesteps: int):
    if schedule == "linear":
        return linear_beta_schedule(timesteps)
    if schedule == "cosine":
        return cosine_beta_schedule(timesteps)
    raise ValueError(f"unknown schedule: {schedule}")

### The `extract` helper

Every diffusion formula looks like `coefficient[t] * image`. But `t` is a *batch* of
timesteps (one per image) and the image is `(B, C, H, W)`, so we need to gather $B$ scalars
and reshape them to `(B, 1, 1, 1)` so they broadcast against the images.

Almost every shape bug in diffusion code lives in this function. Broadcasting a `(B,)`
tensor against `(B, 1, 28, 28)` silently gives you `(B, 1, 28, B)` — no error, garbage results.

> **TODO:** implement `extract`.

In [ ]:
def extract(values: torch.Tensor, t: torch.Tensor, x_shape) -> torch.Tensor:
    """Gather values[t] and reshape for broadcasting against x.

    values:  (T,)              a precomputed schedule quantity
    t:       (B,) int64        one timestep index per image in the batch
    x_shape: e.g. (B, C, H, W)
    returns: (B, 1, 1, 1)
    """
    # TODO (2 lines):
    #   1. pick out values[t]       -> shape (B,)    hint: values.gather(0, t)
    #   2. reshape to (B, 1, 1, 1)  so it broadcasts over C, H, W.
    #      Use len(x_shape) so this works for any rank, not just 4.
    raise NotImplementedError

### Bundling the schedule

`Diffusion` precomputes, once and on the right device, every constant the forward and
reverse processes need. Deriving these by hand is the point of this part:

$$\alpha_t = 1-\beta_t \qquad \bar\alpha_t = \prod_{s\le t}\alpha_s \qquad
\bar\alpha_{t-1} = \text{shift}(\bar\alpha_t) \text{ with } \bar\alpha_{-1} := 1$$

and the variance of the true posterior $q(x_{t-1} \mid x_t, x_0)$, which the sampler needs:

$$\tilde\beta_t = \beta_t \cdot \frac{1 - \bar\alpha_{t-1}}{1 - \bar\alpha_t}$$

We compute in `float64` and cast down at the end — the cumulative product over 1000 steps
loses meaningful precision in `float32`.

> **TODO:** fill in the marked lines.

In [ ]:
class Diffusion:
    """Holds the noise schedule and the forward process q(x_t | x_0)."""

    def __init__(self, timesteps: int, schedule: str = "cosine", device=torch.device("cpu")):
        self.timesteps = timesteps
        self.device = device

        betas = make_betas(schedule, timesteps)                        # (T,) float64

        # TODO (4 lines):
        #   alphas              = 1 - betas
        #   alphas_cumprod      = cumulative product of alphas          (torch.cumprod)
        #   alphas_cumprod_prev = alphas_cumprod shifted right by one, 1.0 prepended,
        #                         still length T   hint: F.pad(x[:-1], (1, 0), value=1.0)
        #   posterior_variance  = betas * (1 - alphas_cumprod_prev) / (1 - alphas_cumprod)
        alphas = ...
        alphas_cumprod = ...
        alphas_cumprod_prev = ...
        posterior_variance = ...

        def reg(x):
            return x.to(device=device, dtype=torch.float32)

        # --- forward process q(x_t | x_0) ---
        self.betas = reg(betas)
        self.alphas = reg(alphas)
        self.alphas_cumprod = reg(alphas_cumprod)
        self.alphas_cumprod_prev = reg(alphas_cumprod_prev)
        self.sqrt_alphas_cumprod = reg(...)                 # TODO: sqrt(alpha_bar_t)
        self.sqrt_one_minus_alphas_cumprod = reg(...)       # TODO: sqrt(1 - alpha_bar_t)

        # --- reverse process p(x_{t-1} | x_t) ---
        self.posterior_variance = reg(posterior_variance)
        # coefficients of the posterior mean, in terms of x0_hat and x_t (see Part 7)
        self.posterior_mean_coef1 = reg(
            betas * alphas_cumprod_prev.sqrt() / (1.0 - alphas_cumprod))
        self.posterior_mean_coef2 = reg(
            (1.0 - alphas_cumprod_prev) * alphas.sqrt() / (1.0 - alphas_cumprod))

    def q_sample(self, x0, t, noise=None):
        """Sample x_t ~ q(x_t | x_0). You will implement this in Part 3."""
        raise NotImplementedError("implemented in Part 3")


diffusion = Diffusion(cfg.timesteps, cfg.schedule, device)
print("betas:", tuple(diffusion.betas.shape),
      "| alpha_bar[0] =", round(diffusion.alphas_cumprod[0].item(), 5),
      "| alpha_bar[-1] =", round(diffusion.alphas_cumprod[-1].item(), 5))

#### ✅ Sanity check — the schedule

In [ ]:
d = diffusion
assert d.betas.shape == (cfg.timesteps,)
assert torch.all(d.betas > 0) and torch.all(d.betas < 1), "betas must live in (0, 1)"
assert torch.all(d.alphas_cumprod[1:] <= d.alphas_cumprod[:-1] + 1e-6), "alpha_bar must decrease"
assert d.alphas_cumprod[0] > 0.99, "there should be almost no noise at t=0"
assert d.alphas_cumprod[-1] < 0.02, "there should be almost only noise at t=T-1"
assert torch.isclose(d.alphas_cumprod_prev[0], torch.tensor(1.0, device=device))
assert torch.allclose(d.alphas_cumprod_prev[1:], d.alphas_cumprod[:-1])
assert torch.allclose(d.sqrt_alphas_cumprod ** 2 + d.sqrt_one_minus_alphas_cumprod ** 2,
                      torch.ones_like(d.betas), atol=1e-5), "the two coefficients must square-sum to 1"
assert torch.all(d.posterior_variance >= 0)
# at t=0 the posterior mean is exactly x0, so coef1 = 1 and coef2 = 0
assert torch.isclose(d.posterior_mean_coef1[0], torch.tensor(1.0, device=device), atol=1e-3)
assert d.posterior_mean_coef2[0].abs() < 1e-3

# how the two schedules compare
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name in ("linear", "cosine"):
    b = make_betas(name, cfg.timesteps).float()
    axes[0].plot(b.numpy(), label=name)
    axes[1].plot(torch.cumprod(1 - b, 0).numpy(), label=name)
axes[0].set_title(r"$\beta_t$  (noise added per step)")
axes[1].set_title(r"$\bar{\alpha}_t$  (signal remaining)")
for ax in axes:
    ax.set_xlabel("t")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("✅ schedule looks right")

## Part 3 — The forward process: jump to any $t$ in one step

Applying $T$ small noising steps one at a time would make training hopelessly slow. The key
algebraic result of DDPM is that composing all those Gaussians has a closed form:

$$q(x_t \mid x_0) = \mathcal{N}\!\left(x_t;\ \sqrt{\bar{\alpha}_t}\, x_0,\ (1-\bar{\alpha}_t) I\right)$$

which, via the reparameterisation trick, means

$$\boxed{\;x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon,
\qquad \varepsilon \sim \mathcal{N}(0, I)\;}$$

So for *any* $t$ we get a training example in one line — and $\varepsilon$, the thing we just
sampled, is exactly the label the network will be asked to predict. That is the whole trick
that makes diffusion training cheap.

Sanity intuition: at $t=0$, $\bar\alpha \approx 1$ so $x_t \approx x_0$; at $t=T$,
$\bar\alpha \approx 0$ so $x_t \approx \varepsilon$, i.e. standard normal noise.

> **TODO:** implement `q_sample`.

In [ ]:
def q_sample(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor | None = None):
    """Sample x_t ~ q(x_t | x_0) = N(sqrt(a_bar_t) * x0, (1 - a_bar_t) I).

    x0:    (B, C, H, W) clean images in [-1, 1]
    t:     (B,) int64 timesteps
    noise: (B, C, H, W), or None to sample fresh noise
    """
    if noise is None:
        noise = torch.randn_like(x0)
    # TODO (3 lines): pull sqrt(a_bar_t) and sqrt(1 - a_bar_t) out of `self` with `extract`,
    # then return  sqrt(a_bar_t) * x0 + sqrt(1 - a_bar_t) * noise
    raise NotImplementedError


Diffusion.q_sample = q_sample   # attach to the class defined above

#### ✅ Sanity check — the forward process

In [ ]:
x0 = x_batch.to(device)

# t = 0 should barely change the image
t0 = torch.zeros(x0.shape[0], dtype=torch.long, device=device)
assert (diffusion.q_sample(x0, t0) - x0).abs().mean() < 0.15, "t=0 should be nearly noiseless"

# t = T-1 should be ~ N(0, 1)
tT = torch.full((x0.shape[0],), cfg.timesteps - 1, dtype=torch.long, device=device)
xT = diffusion.q_sample(x0, tT)
print(f"x_T  mean = {xT.mean():+.3f} (want ~0)   std = {xT.std():.3f} (want ~1)")
assert abs(xT.mean().item()) < 0.1 and abs(xT.std().item() - 1.0) < 0.1

# given the same noise, q_sample must be deterministic
eps = torch.randn_like(x0)
t = torch.randint(0, cfg.timesteps, (x0.shape[0],), device=device)
assert torch.allclose(diffusion.q_sample(x0, t, eps), diffusion.q_sample(x0, t, eps))

# and it must use a *per-image* t, not one t for the whole batch
mixed = diffusion.q_sample(x0[:2], torch.tensor([0, cfg.timesteps - 1], device=device), eps[:2])
assert (mixed[0] - x0[0]).abs().mean() < (mixed[1] - x0[1]).abs().mean(), \
    "image 0 (t=0) should be much closer to the original than image 1 (t=T-1)"
print("✅ q_sample looks right")

In [ ]:
# Watch a digit dissolve.
img = x_batch[:1].to(device)
steps = torch.linspace(0, cfg.timesteps - 1, 10).long().to(device)
noised = torch.cat([diffusion.q_sample(img, s.view(1)) for s in steps])
show(noised, title=f"forward process, t = {steps.tolist()}", nrow=10, figsize=(12, 2))

## Part 4 — The model: a U-Net that predicts noise

We need a network $\varepsilon_\theta(x_t, t)$ that maps a noisy image plus a timestep to a
prediction of the noise that was added. Requirements:

1. **Output shape = input shape.** It predicts a full-resolution noise image → U-Net.
2. **It must know $t$.** The same network handles "barely noisy" and "pure static", which
   are completely different jobs. We inject $t$ as an embedding added inside every residual block.
3. **Global context.** A digit's strokes must agree across the whole image → one
   self-attention block at the lowest resolution.

### 4a. Timestep embeddings

$t$ is a single integer. Feeding it as a raw number makes it hard for the network to
distinguish nearby timesteps. Instead we use the same sinusoidal embedding as the
Transformer, which spreads $t$ over many frequencies:

$$\text{emb}(t)_{2i} = \sin\!\left(\frac{t}{10000^{2i/d}}\right), \qquad
\text{emb}(t)_{2i+1} = \cos\!\left(\frac{t}{10000^{2i/d}}\right)$$

Implementation note: compute the frequencies as
`exp(-log(10000) * arange(half) / half)`, multiply by `t`, then concatenate `sin` and `cos`.

> **TODO:** implement `timestep_embedding`.

In [ ]:
def timestep_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """Sinusoidal embedding of a batch of timesteps.

    t:   (B,) int64 or float
    dim: embedding size (even)
    returns: (B, dim)
    """
    half = dim // 2
    # TODO (3 lines):
    #   freqs = exp(-log(10000) * arange(half) / half)          -> (half,)   [same device as t]
    #   args  = t (as float, shape (B, 1)) * freqs (shape (1, half))  -> (B, half)
    #   return concat([sin(args), cos(args)], dim=-1)           -> (B, dim)
    raise NotImplementedError

#### ✅ Sanity check — timestep embeddings

In [ ]:
emb = timestep_embedding(torch.arange(cfg.timesteps, device=device), 64)
assert emb.shape == (cfg.timesteps, 64)
assert torch.isfinite(emb).all()
assert not torch.allclose(emb[0], emb[1]), "adjacent timesteps must differ"
assert emb.abs().max() <= 1.0 + 1e-5, "sin/cos are bounded by 1"
# neighbouring timesteps should be more similar than distant ones
near, far = cfg.timesteps // 10, cfg.timesteps * 9 // 10
sim = lambda a, b: F.cosine_similarity(a[None], b[None]).item()
assert sim(emb[near], emb[near + 1]) > sim(emb[near], emb[far])

plt.figure(figsize=(9, 2.5))
plt.imshow(emb.cpu().T.numpy(), aspect="auto", cmap="RdBu")
plt.xlabel("timestep t")
plt.ylabel("embedding dim")
plt.title("sinusoidal timestep embeddings")
plt.colorbar()
plt.show()
print("✅ timestep_embedding looks right")

### 4b. Building blocks

Three small modules. `AttentionBlock`, `Downsample` and `Upsample` are given; you write
the residual block, which is where the timestep actually enters the network.

**`ResBlock(x, t_emb)`** does:

```
h = conv1(silu(norm1(x)))                      # 3x3 conv
h = h + time_proj(silu(t_emb))[:, :, None, None]   # broadcast a per-channel shift over H, W
h = conv2(dropout(silu(norm2(h))))             # 3x3 conv
return h + skip(x)                             # 1x1 conv on the skip iff channels change
```

That single `+ time_proj(...)` line is how the network is told "this image has 30% noise
in it". Note `[:, :, None, None]`: the projection gives `(B, C)`, which must be reshaped to
`(B, C, 1, 1)` to broadcast over spatial dims.

We use `GroupNorm` rather than `BatchNorm`: sampling runs with batch statistics that differ
wildly from training, and BatchNorm would break badly.

In [ ]:
class AttentionBlock(nn.Module):
    """Multi-head self-attention over spatial positions. Provided for you."""

    def __init__(self, channels: int, num_heads: int = 4):
        super().__init__()
        self.num_heads = num_heads
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
        nn.init.zeros_(self.proj.weight)      # start as an identity function
        nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        B, C, H, W = x.shape
        qkv = self.qkv(self.norm(x))
        q, k, v = qkv.reshape(B, 3, self.num_heads, C // self.num_heads, H * W).unbind(1)
        q, k, v = (z.transpose(-2, -1) for z in (q, k, v))   # (B, heads, HW, head_dim)
        out = F.scaled_dot_product_attention(q, k, v)
        out = out.transpose(-2, -1).reshape(B, C, H, W)
        return x + self.proj(out)


class Downsample(nn.Module):
    """Halve the resolution with a strided conv. Provided for you."""

    def __init__(self, channels: int):
        super().__init__()
        self.op = nn.Conv2d(channels, channels, 3, stride=2, padding=1)

    def forward(self, x):
        return self.op(x)


class Upsample(nn.Module):
    """Double the resolution: nearest-neighbour + conv (avoids checkerboard artifacts)."""

    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode="nearest"))

In [ ]:
class ResBlock(nn.Module):
    """Residual block conditioned on the timestep embedding."""

    def __init__(self, in_ch: int, out_ch: int, t_dim: int, dropout: float = 0.1,
                 attn: bool = False):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)

        self.time_proj = nn.Linear(t_dim, out_ch)

        self.norm2 = nn.GroupNorm(8, out_ch)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.attn = AttentionBlock(out_ch) if attn else nn.Identity()

    def forward(self, x, t_emb):
        # TODO (5 lines), using the modules defined above and F.silu:
        #   h = conv1(silu(norm1(x)))
        #   h = h + time_proj(silu(t_emb)) reshaped from (B, C) to (B, C, 1, 1)
        #   h = conv2(dropout(silu(norm2(h))))
        #   h = h + skip(x)
        #   return self.attn(h)      <- Identity when attn=False
        raise NotImplementedError

### 4c. The U-Net

The architecture, at `base_channels=64` and `channel_mults=(1, 2, 2)`:

```
28x28x1  ──stem──► 28x28x64 ─┐
   ResBlock x2                │ skips
   Downsample ► 14x14x64 ─────┤
   ResBlock x2 (128ch)         │
   Downsample ► 7x7x128 ──────┤
   ResBlock x2 + attention     │
        ┌──── middle: ResBlock, ResBlock (attention) ────┐
   ResBlock x3 (concat skip) ◄─┘
   Upsample ► 14x14
   ResBlock x3 (concat skip)
   Upsample ► 28x28
   ResBlock x3 (concat skip)
   GroupNorm ► SiLU ► conv ► 28x28x1   (predicted noise)
```

`__init__` is written for you — the channel bookkeeping is fiddly and not the point.
**You write `forward`**, which is where the two ideas that define a U-Net live:

- every down-path output is **pushed onto a stack** `hs`,
- every up-path block **pops one off and concatenates it along the channel dim** before running.

Get the stack order wrong and you'll see a channel-count mismatch error immediately.

Note the final conv is zero-initialised, so an untrained model predicts exactly zero noise.
That makes the loss at initialisation *exactly* $\mathbb{E}[\varepsilon^2] = 1$ — a free
diagnostic we use in the next sanity check.

`y` (a class label) is ignored for now; it powers the conditional bonus in Part 10.

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, channel_mults=(1, 2, 2),
                 num_res_blocks=2, attn_resolutions=(7,), image_size=28,
                 dropout=0.1, num_classes: int | None = None):
        super().__init__()
        self.base_channels = base_channels
        self.num_classes = num_classes
        t_dim = base_channels * 4

        # timestep (and optional class) conditioning
        self.time_mlp = nn.Sequential(
            nn.Linear(base_channels, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim)
        )
        if num_classes is not None:
            # one extra row is the "null" / unconditional token, used by CFG
            self.label_emb = nn.Embedding(num_classes + 1, t_dim)

        self.stem = nn.Conv2d(in_channels, base_channels, 3, padding=1)

        # ---- down path ----
        ch = base_channels
        res = image_size
        skip_chans = [ch]
        self.down_levels = nn.ModuleList()
        for i, mult in enumerate(channel_mults):
            out_ch = base_channels * mult
            blocks = nn.ModuleList()
            for _ in range(num_res_blocks):
                blocks.append(ResBlock(ch, out_ch, t_dim, dropout, attn=res in attn_resolutions))
                ch = out_ch
                skip_chans.append(ch)
            if i < len(channel_mults) - 1:
                down = Downsample(ch)
                skip_chans.append(ch)
                res //= 2
            else:
                down = nn.Identity()
            self.down_levels.append(nn.ModuleList([blocks, down]))

        # ---- middle ----
        self.mid1 = ResBlock(ch, ch, t_dim, dropout, attn=True)
        self.mid2 = ResBlock(ch, ch, t_dim, dropout, attn=False)

        # ---- up path ----
        self.up_levels = nn.ModuleList()
        for i, mult in reversed(list(enumerate(channel_mults))):
            out_ch = base_channels * mult
            blocks = nn.ModuleList()
            for _ in range(num_res_blocks + 1):   # +1 to consume the downsample skip
                blocks.append(ResBlock(ch + skip_chans.pop(), out_ch, t_dim, dropout,
                                       attn=res in attn_resolutions))
                ch = out_ch
            if i > 0:
                up = Upsample(ch)
                res *= 2
            else:
                up = nn.Identity()
            self.up_levels.append(nn.ModuleList([blocks, up]))
        assert not skip_chans, "every skip must be consumed exactly once"

        # ---- output ----
        self.out_norm = nn.GroupNorm(8, ch)
        self.out_conv = nn.Conv2d(ch, in_channels, 3, padding=1)
        nn.init.zeros_(self.out_conv.weight)   # predict zero noise at initialisation
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, x, t, y=None):
        """x: (B, C, H, W) noisy images | t: (B,) timesteps | y: (B,) labels or None
        returns: (B, C, H, W) predicted noise."""
        # 1. Build the conditioning vector.
        #    TODO: t_emb = self.time_mlp(timestep_embedding(t, self.base_channels))
        #          if y is not None and self.num_classes is not None:
        #              add self.label_emb(y) to t_emb
        t_emb = ...

        # 2. Stem, and start the skip stack.
        h = self.stem(x)
        hs = [h]

        # 3. Down path.
        #    TODO: for blocks, down in self.down_levels:
        #              run every block as h = block(h, t_emb) and APPEND h to hs each time
        #              if `down` is not an nn.Identity: h = down(h) and append h to hs too
        #    (use `isinstance(down, nn.Identity)` to test)

        # 4. Middle: two ResBlocks, mid1 then mid2.
        #    TODO

        # 5. Up path.
        #    TODO: for blocks, up in self.up_levels:
        #              for each block: h = block(torch.cat([h, hs.pop()], dim=1), t_emb)
        #              then h = up(h)          <- Identity at the last level
        #    The concat is what makes this a U-Net: pop from the END of hs (LIFO).

        # 6. Output head.
        #    TODO: return self.out_conv(F.silu(self.out_norm(h)))
        raise NotImplementedError

#### ✅ Sanity check — the U-Net

In [ ]:
model = UNet(
    in_channels=cfg.channels,
    base_channels=cfg.base_channels,
    channel_mults=cfg.channel_mults,
    num_res_blocks=cfg.num_res_blocks,
    attn_resolutions=cfg.attn_resolutions,
    image_size=cfg.image_size,
    dropout=cfg.dropout,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params/1e6:.2f}M parameters")

xt = torch.randn(4, cfg.channels, cfg.image_size, cfg.image_size, device=device)
t = torch.randint(0, cfg.timesteps, (4,), device=device)
with torch.no_grad():
    out = model(xt, t)

assert out.shape == xt.shape, f"output {tuple(out.shape)} must match input {tuple(xt.shape)}"
assert torch.isfinite(out).all()
assert out.abs().max() == 0, "zero-init output layer means an untrained model predicts 0"

# the output must actually depend on t: perturb only t and check the change
model.eval()
with torch.no_grad():
    model.out_conv.weight.normal_(0, 0.05)   # temporarily un-zero the head so we can see a signal
    a = model(xt, torch.zeros(4, dtype=torch.long, device=device))
    b = model(xt, torch.full((4,), cfg.timesteps - 1, dtype=torch.long, device=device))
    nn.init.zeros_(model.out_conv.weight)
assert (a - b).abs().mean() > 1e-6, "the model ignores t — is the time embedding wired in?"
model.train()
print("✅ U-Net looks right")

## Part 5 — The training objective

The full variational bound on the likelihood is a sum of KL divergences between Gaussians.
Ho et al. showed that if you reparameterise the model to predict $\varepsilon$ and drop the
per-timestep weighting, the whole thing collapses to a plain MSE — and works *better* in practice:

$$L_\text{simple} = \mathbb{E}_{x_0,\ t \sim \mathcal{U}\{0,T-1\},\ \varepsilon \sim \mathcal{N}(0,I)}
\left[\; \left\lVert \varepsilon - \varepsilon_\theta\!\left(\sqrt{\bar\alpha_t} x_0 +
\sqrt{1-\bar\alpha_t}\,\varepsilon,\ t\right) \right\rVert^2 \right]$$

One training step is four lines:

1. sample a random $t$ per image,
2. sample noise $\varepsilon$,
3. build $x_t$ with `q_sample`,
4. MSE between $\varepsilon$ and the model's prediction.

Sampling $t$ *uniformly per image* matters: it trains all noise levels at once, and it is
why one pass over the data teaches the model the whole reverse trajectory.

> **TODO:** implement `p_losses`.

In [ ]:
def p_losses(model, diffusion, x0, t, y=None, noise=None):
    """The DDPM training loss for one batch.

    x0: (B, C, H, W) clean images | t: (B,) timesteps | y: (B,) labels or None
    """
    if noise is None:
        noise = torch.randn_like(x0)
    # TODO (3 lines):
    #   x_t = the noisy image at timestep t  (diffusion.q_sample, pass the SAME noise)
    #   predicted_noise = model(x_t, t, y)
    #   return the mean-squared error between the prediction and the true noise
    raise NotImplementedError

#### ✅ Sanity check — the loss

In [ ]:
x0 = x_batch.to(device)
t = torch.randint(0, cfg.timesteps, (x0.shape[0],), device=device)
loss = p_losses(model, diffusion, x0, t)
print(f"loss at initialisation: {loss.item():.4f}  (want ~1.0)")

# The head is zero-initialised, so the model predicts 0 and the loss is E[eps^2] = 1.
assert abs(loss.item() - 1.0) < 0.1, "should be ~1.0 — are you comparing against the noise?"
assert loss.requires_grad, "loss must be differentiable"

# A common bug: predicting x0 instead of eps. That loss would look very different.
loss.backward()
assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters()), \
    "no gradients reached the parameters"
model.zero_grad(set_to_none=True)
print("✅ p_losses looks right")

## Part 6 — Training

Standard supervised training on the loss above. Two details worth knowing:

- **EMA of the weights.** Diffusion sample quality is famously sensitive to weight noise.
  Keeping an exponential moving average of the parameters ($\theta_\text{ema} \leftarrow
  0.999\,\theta_\text{ema} + 0.001\,\theta$) and sampling from *that* is close to free and
  usually gives cleaner, more consistent samples. It is written for you below — including the
  decay warm-up, without which the EMA actively hurts on runs shorter than a few thousand steps.
- **Gradient clipping** at norm 1.0, which keeps the occasional high-noise outlier batch
  from wrecking the run.

> **TODO:** fill in the training step inside `train`.

In [ ]:
class EMA:
    """Exponential moving average of model parameters. Provided for you.

    The decay is warmed up as min(decay, (1 + step) / (10 + step)), exactly as in the
    reference DDPM implementation. Without it, a 0.999 average has a ~1000-step memory, so
    on a short run the EMA weights are still mostly the random initialisation — and the
    "improved" samples come out visibly worse than the raw ones.
    """

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.step = 0
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = min(self.decay, (1 + self.step) / (10 + self.step))
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(d).add_(v.detach(), alpha=1 - d)
            else:
                self.shadow[k].copy_(v)

    def copy_to(self, model):
        model.load_state_dict(self.shadow)

In [ ]:
def train(model, loader, diffusion, cfg, epochs=None, cond_dropout=0.0):
    """Train the noise predictor. Returns (ema, loss_history).

    cond_dropout > 0 replaces labels with the null token that fraction of the time
    (used only by the classifier-free guidance bonus in Part 10).
    """
    epochs = cfg.epochs if epochs is None else epochs
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
    ema = EMA(model, cfg.ema_decay)
    history = []

    for epoch in range(epochs):
        model.train()
        pbar = tqdm(loader, desc=f"epoch {epoch + 1}/{epochs}", leave=False)
        running = 0.0
        for step, (x0, labels) in enumerate(pbar):
            x0 = x0.to(device)
            B = x0.shape[0]

            if cond_dropout > 0:
                y = labels.to(device)
                drop = torch.rand(B, device=device) < cond_dropout
                y = torch.where(drop, torch.full_like(y, model.num_classes), y)
            else:
                y = None

            # --- one training step ---
            # TODO (7 lines):
            #   t = a random timestep per image: torch.randint(0, T, (B,), device=device).long()
            #   loss = p_losses(model, diffusion, x0, t, y)
            #   zero the optimiser grads (set_to_none=True)
            #   loss.backward()
            #   clip grads: torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            #   opt.step()
            #   ema.update(model)
            raise NotImplementedError
            # --- end training step ---

            history.append(loss.item())
            running += loss.item()
            if step % 20 == 0:
                pbar.set_postfix(loss=f"{running / (step + 1):.4f}")

        print(f"epoch {epoch + 1}/{epochs}  mean loss {running / len(loader):.4f}")

    return ema, history

### Run it

Roughly 1–2 min/epoch on a modern GPU, 10–20 min/epoch on CPU (drop `epochs`, set
`train_subset`, and lower `timesteps` to ~400 if you are on CPU).

**What to expect:** the loss drops fast to ~0.1 in the first epoch, then improves very slowly
and noisily. That is normal — the loss averages over all noise levels, and the easy high-noise
timesteps dominate it. Loss curves are a poor proxy for sample quality here; look at the samples.

In [ ]:
ema, history = train(model, train_loader, diffusion, cfg)

plt.figure(figsize=(9, 3))
plt.plot(history, alpha=0.3, label="per step")
if len(history) > 50:
    k = 50
    smooth = np.convolve(history, np.ones(k) / k, mode="valid")
    plt.plot(range(k - 1, len(history)), smooth, label=f"{k}-step mean")
plt.xlabel("step")
plt.ylabel("MSE loss")
plt.ylim(0, max(0.5, float(np.percentile(history, 99))))
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Save your work so you never have to retrain.
torch.save({"model": model.state_dict(), "ema": ema.shadow, "cfg": cfg.__dict__},
           "ddpm_mnist.pt")
print("saved ddpm_mnist.pt")

# To reload later:
# ckpt = torch.load("ddpm_mnist.pt", map_location=device, weights_only=False)
# model.load_state_dict(ckpt["model"])

## Part 7 — Sampling: turning noise into digits

Training taught the model $\varepsilon_\theta(x_t, t)$. To *generate*, we start from
$x_T \sim \mathcal{N}(0, I)$ and walk backwards, one denoising step at a time.

Bayes' rule gives the true posterior in closed form — *if* you know the clean image:

$$q(x_{t-1} \mid x_t, x_0) = \mathcal{N}\!\left(x_{t-1};\ \tilde\mu_t,\ \tilde\beta_t I\right),
\qquad \tilde\mu_t = \underbrace{\frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}}_{\text{coef1}} x_0
+ \underbrace{\frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}}_{\text{coef2}} x_t$$

We don't have $x_0$ at sampling time — but the noise prediction gives us an estimate of it,
by solving the forward equation for $x_0$:

$$\hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\;\varepsilon_\theta(x_t, t)}{\sqrt{\bar\alpha_t}}$$

So each reverse step is four moves:

1. **predict** the noise in $x_t$,
2. **solve** for the clean image $\hat{x}_0$ it implies, and **clamp** it to $[-1, 1]$,
3. **step** partway back toward it: $\tilde\mu_t = \text{coef1}\cdot\hat{x}_0 + \text{coef2}\cdot x_t$,
4. **add** fresh noise $\sqrt{\tilde\beta_t}\,z$ — except at the final step ($t=0$), where we
   return the mean. Forgetting to skip the noise at $t=0$ leaves visible speckle on every
   sample; adding no noise at any step collapses everything toward one blurry average digit.

> **Why not the one-line form?** You will see this step written compactly as
> $x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\varepsilon_\theta\right) + \sqrt{\tilde\beta_t}z$.
> It is algebraically identical and it is the form printed in the DDPM paper — but it leaves
> nowhere to put the clamp. With a cosine schedule the last timesteps have
> $\alpha_t \approx 0.001$, so that $1/\sqrt{\alpha_t}$ factor amplifies the model's error by
> ~30x *per step* and the trajectory blows up long before it reaches $t=0$. Every production
> implementation clamps $\hat{x}_0$. Keeping it explicit also sets up DDIM in Part 9, which is
> written entirely in terms of $\hat{x}_0$.

This runs the network $T$ times per batch of images, which is why diffusion sampling is slow.
Part 9 fixes that.

> **TODO:** implement `p_sample` and the `sample` loop.

In [ ]:
@torch.no_grad()
def p_sample(model, diffusion, x, t, t_index, y=None, clip=True):
    """One reverse step: sample x_{t-1} ~ p(x_{t-1} | x_t).

    x:       (B, C, H, W) current noisy images
    t:       (B,) int64, all equal to t_index
    t_index: python int, the current timestep (T-1 down to 0)
    """
    # TODO — the four moves from the markdown above. Everything you need is on
    # `diffusion`; pull each one out with `extract(..., t, x.shape)`.
    #   1. eps_theta = model(x, t, y)
    #   2. x0_pred = (x - sqrt_one_minus_alphas_cumprod_t * eps_theta) / sqrt_alphas_cumprod_t
    #      then, if clip: clamp it to [-1, 1]
    #   3. mean = posterior_mean_coef1_t * x0_pred + posterior_mean_coef2_t * x
    #   4. if t_index == 0: return mean          <- no noise on the very last step!
    #      otherwise return mean + sqrt(posterior_variance_t) * randn_like(x)
    raise NotImplementedError


@torch.no_grad()
def sample(model, diffusion, n=16, y=None, return_trajectory=False, shape=None):
    """Generate n images by running the reverse process from pure noise."""
    model.eval()
    shape = shape or (n, cfg.channels, cfg.image_size, cfg.image_size)

    # TODO:
    #   x = pure Gaussian noise of `shape`, on `device`
    #   for i in reversed(range(diffusion.timesteps)):        # T-1, T-2, ..., 0
    #       t = torch.full((shape[0],), i, device=device, dtype=torch.long)
    #       x = p_sample(model, diffusion, x, t, i, y)
    #   (wrap the loop in tqdm(..., total=diffusion.timesteps) so you can watch it)
    #   if return_trajectory: also stash x.clone() every T//10 steps into a list `traj`
    raise NotImplementedError

    model.train()
    return (x, traj) if return_trajectory else x

#### ✅ Sanity check — the sampler

In [ ]:
model.eval()
torch.manual_seed(0)
xt = torch.randn(2, cfg.channels, cfg.image_size, cfg.image_size, device=device)

# 1. At t=0, coef1 = 1 and coef2 = 0, so the step must return exactly the clamped x0
#    estimate — no noise added. This checks the whole formula without needing a trained model.
t0 = torch.zeros(2, dtype=torch.long, device=device)
with torch.no_grad():
    eps0 = model(xt, t0)
    ab0 = diffusion.alphas_cumprod[0]
    expected = ((xt - (1 - ab0).sqrt() * eps0) / ab0.sqrt()).clamp(-1, 1)
    got = p_sample(model, diffusion, xt, t0, 0)
assert torch.allclose(got, expected, atol=1e-4), \
    "the t=0 step must return the clamped x0 estimate — no noise term; check the coefficients"

# 2. Every other step must add fresh noise, so two calls differ.
k = cfg.timesteps // 2
tk = torch.full((2,), k, dtype=torch.long, device=device)
with torch.no_grad():
    assert not torch.allclose(p_sample(model, diffusion, xt, tk, k),
                              p_sample(model, diffusion, xt, tk, k)), \
        "steps with t > 0 must add fresh noise"
model.train()

# 3. The full loop: shape, finiteness, range.
probe = sample(model, diffusion, n=4)
assert probe.shape == (4, cfg.channels, cfg.image_size, cfg.image_size)
assert torch.isfinite(probe).all(), "NaNs: check for a division by ~0 or a missing sqrt"
print(f"sample range: [{probe.min():.2f}, {probe.max():.2f}]   (real data is [-1, 1])")
assert probe.abs().max() <= 1.0 + 1e-4, \
    "the last step returns the clamped x0 estimate, so samples must land inside [-1, 1]"
print("✅ sampler looks right")

### The payoff

We sample from the EMA weights, which are usually a little cleaner and more consistent than
the raw ones. Both are shown so you can judge for yourself — on a short run the two are often
close, and that is worth noticing rather than assuming EMA always wins.

In [ ]:
ema_model = UNet(
    in_channels=cfg.channels, base_channels=cfg.base_channels,
    channel_mults=cfg.channel_mults, num_res_blocks=cfg.num_res_blocks,
    attn_resolutions=cfg.attn_resolutions, image_size=cfg.image_size, dropout=cfg.dropout,
).to(device)
ema.copy_to(ema_model)

torch.manual_seed(1234)
raw_samples = sample(model, diffusion, n=32)
torch.manual_seed(1234)
ema_samples = sample(ema_model, diffusion, n=32)

show(raw_samples, title="samples — raw weights", nrow=8, figsize=(8, 4))
show(ema_samples, title="samples — EMA weights", nrow=8, figsize=(8, 4))

In [ ]:
# Watch the reverse process run: noise on the left, a digit on the right.
_, traj = sample(ema_model, diffusion, n=8, return_trajectory=True)
strip = torch.cat([step[:8] for step in traj])
show(strip, title="reverse process (left = noisy, right = final)",
     nrow=8, figsize=(8, 1.2 * len(traj)))

## Part 8 — Is it actually generating, or just memorising?

The failure mode that a loss curve will never show you: a generative model that has
memorised training images. The cheap check is a nearest-neighbour lookup — for each sample,
find the closest training image in pixel space. If your samples are near-duplicates of their
neighbours, the model has memorised rather than generalised.

With 60k images, 8 epochs and ~7M parameters you should see *similar but clearly distinct*
digits. Blurry, broken, or half-formed digits mean undertraining, not memorisation.

In [ ]:
@torch.no_grad()
def nearest_neighbours(samples, dataset, n_search=10000):
    """For each sample, find its closest training image by L2 distance."""
    imgs = torch.stack([dataset[i][0] for i in range(min(n_search, len(dataset)))]).to(device)
    flat_db = imgs.flatten(1)
    flat_q = samples.to(device).flatten(1)
    dists = torch.cdist(flat_q, flat_db)              # (n_samples, n_search)
    idx = dists.argmin(dim=1)
    return imgs[idx].cpu(), dists.min(dim=1).values.cpu()


queries = ema_samples[:8]
neigh, dist = nearest_neighbours(queries, train_set)
show(torch.cat([queries.cpu(), neigh]), nrow=8, figsize=(8, 2.4),
     title="top: generated   bottom: nearest training image")
print("L2 distances:", [round(d.item(), 2) for d in dist])

In [ ]:
# Two more quick diagnostics.
print(f"generated  mean {ema_samples.mean():+.3f}  std {ema_samples.std():.3f}")
print(f"real data  mean {x_batch.mean():+.3f}  std {x_batch.std():.3f}")

plt.figure(figsize=(9, 3))
plt.hist(x_batch.flatten().numpy(), bins=60, alpha=0.6, density=True, label="real")
plt.hist(ema_samples.cpu().flatten().numpy(), bins=60, alpha=0.6, density=True, label="generated")
plt.title("pixel value distribution")
plt.legend()
plt.show()

## Part 9 — *Bonus:* DDIM, or sampling 20x faster

DDPM sampling needs all $T$ steps because each step injects fresh noise. DDIM
(Song et al., 2021) reuses the *same trained model* with a **deterministic** update that can
skip timesteps — 50 steps instead of 1000, at slightly lower diversity.

The trick uses the same $\hat{x}_0$ estimate you built in Part 7 — from $x_t$ and the predicted
noise, recover the clean image,

$$\hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\;\varepsilon_\theta(x_t,t)}{\sqrt{\bar\alpha_t}}$$

then re-noise it to the *next* timestep in your (shortened) schedule:

$$x_{t_\text{prev}} = \sqrt{\bar\alpha_{t_\text{prev}}}\;\hat{x}_0 +
\sqrt{1-\bar\alpha_{t_\text{prev}}}\;\varepsilon_\theta(x_t,t)$$

No randomness after $x_T$: the same starting noise always gives the same image, which makes
DDIM latents interpolatable.

> **TODO:** implement `ddim_sample`. Clamp $\hat{x}_0$ to $[-1, 1]$ exactly as in Part 7.

In [ ]:
@torch.no_grad()
def ddim_sample(model, diffusion, n=16, steps=50, y=None, eta=0.0, clip=True):
    """Deterministic DDIM sampling with `steps` << T network evaluations."""
    model.eval()
    shape = (n, cfg.channels, cfg.image_size, cfg.image_size)
    x = torch.randn(shape, device=device)

    # a strictly decreasing subsequence of timesteps, e.g. [999, 979, ..., 19, 0]
    times = torch.linspace(diffusion.timesteps - 1, 0, steps).long().tolist()

    # TODO: for each pair (t_cur, t_prev) of consecutive entries in `times`:
    #   ab_t     = extract(diffusion.alphas_cumprod, t,      x.shape)
    #   ab_prev  = extract(diffusion.alphas_cumprod, t_prev, x.shape)
    #   eps      = model(x, t, y)
    #   x0_pred  = (x - sqrt(1 - ab_t) * eps) / sqrt(ab_t)      # then clamp to [-1, 1]
    #   x        = sqrt(ab_prev) * x0_pred + sqrt(1 - ab_prev) * eps
    # On the final entry there is no t_prev: set x = x0_pred and stop.
    raise NotImplementedError

    model.train()
    return x


torch.manual_seed(0)
fast = ddim_sample(ema_model, diffusion, n=32, steps=50)
show(fast, title="DDIM, 50 steps (vs 1000 for DDPM)", nrow=8, figsize=(8, 4))

## Part 10 — *Bonus:* class conditioning + classifier-free guidance

So far the model generates *a* digit; we cannot ask for a 7. Two changes fix that:

1. **Conditioning.** Add a label embedding to the timestep embedding (the `UNet` already
   supports this — pass `num_classes=10` and a `y` tensor). During training, randomly replace
   the label with a **null token** ~10% of the time, so one network learns both the
   conditional and unconditional score.
2. **Classifier-free guidance** (Ho & Salimans, 2022). At sampling time, run the model twice
   and extrapolate away from the unconditional prediction:

$$\tilde\varepsilon = \varepsilon_\theta(x_t, t, \varnothing) +
w \cdot \left(\varepsilon_\theta(x_t, t, y) - \varepsilon_\theta(x_t, t, \varnothing)\right)$$

$w = 1$ is ordinary conditional sampling; $w \approx 3$ gives cleaner, more prototypical
digits at the cost of diversity. This one trick is why text-to-image models follow prompts
as well as they do.

Training a second model doubles your compute — treat this section as optional.

In [ ]:
cond_model = UNet(
    in_channels=cfg.channels, base_channels=cfg.base_channels,
    channel_mults=cfg.channel_mults, num_res_blocks=cfg.num_res_blocks,
    attn_resolutions=cfg.attn_resolutions, image_size=cfg.image_size,
    dropout=cfg.dropout, num_classes=10,
).to(device)

cond_ema, cond_history = train(
    cond_model, train_loader, diffusion, cfg,
    epochs=cfg.epochs, cond_dropout=0.1,     # 10% of labels -> null token
)

cond_ema_model = UNet(
    in_channels=cfg.channels, base_channels=cfg.base_channels,
    channel_mults=cfg.channel_mults, num_res_blocks=cfg.num_res_blocks,
    attn_resolutions=cfg.attn_resolutions, image_size=cfg.image_size,
    dropout=cfg.dropout, num_classes=10,
).to(device)
cond_ema.copy_to(cond_ema_model)

In [ ]:
@torch.no_grad()
def ddim_sample_cfg(model, diffusion, labels, steps=50, guidance=3.0, clip=True):
    """DDIM sampling with classifier-free guidance.

    labels: (B,) int64 class labels in [0, 9]
    guidance: w. 1.0 = plain conditional, higher = stronger prompt adherence.
    """
    # TODO: copy your ddim_sample and change exactly one thing — how `eps` is computed:
    #   null = torch.full_like(labels, model.num_classes)
    #   eps_cond   = model(x, t, labels)
    #   eps_uncond = model(x, t, null)
    #   eps = eps_uncond + guidance * (eps_cond - eps_uncond)
    # Efficiency tip: do it in ONE forward pass by concatenating the two branches
    # along the batch dim and splitting the result with .chunk(2).
    raise NotImplementedError


labels = torch.arange(10, device=device).repeat_interleave(8)   # 8 of each digit
for w in (1.0, 3.0):
    imgs = ddim_sample_cfg(cond_ema_model, diffusion, labels, steps=50, guidance=w)
    show(imgs, title=f"one row per digit — guidance w = {w}", nrow=8, figsize=(8, 10))

## Wrap-up

You have built, from scratch: a noise schedule, the closed-form forward process, a
timestep-conditioned U-Net, the $\varepsilon$-prediction objective, a training loop with EMA,
an ancestral sampler, a fast deterministic sampler, and classifier-free guidance. That is the
complete skeleton of Stable Diffusion — what the real thing adds on top is mainly
(a) running in the latent space of a VAE instead of on pixels, (b) text embeddings instead of
a class label, and (c) a great deal of scale.

### Questions to discuss as a group

1. Why is $\varepsilon$-prediction preferred to predicting $x_0$ directly, when the two are
   algebraically equivalent given $x_t$ and $t$? (Hint: think about what each target looks
   like at $t \approx T$, and what the loss weighting implies.)
2. The training loss is nearly flat after epoch 1 but the samples keep improving. What is the
   loss averaging over, and what would a more informative metric look like?
3. Classifier-free guidance with $w > 1$ produces samples that no longer come from the
   learned distribution. Why is that usually an improvement, and what does it cost?
4. DDIM with 50 steps is 20x faster and barely worse. What did we give up?
5. Where would the U-Net need to change for 3-channel 64×64 images? For text conditioning?

### Things to try next

- Swap the cosine schedule for linear and re-train. Watch where the samples degrade, and
  connect that back to the $\bar\alpha_t$ plot in Part 2.
- Predict $x_0$ or the "v" parameterisation instead of $\varepsilon$, and compare.
- Interpolate between two DDIM latents (`slerp` between two `x_T` tensors) and decode the path.
- Add self-attention at 14×14 as well, and see whether the samples improve enough to pay for it.
- Move to FashionMNIST or CIFAR-10 (`in_channels=3`, `channel_mults=(1, 2, 2, 2)`).

### References

- Ho, Jain, Abbeel — *Denoising Diffusion Probabilistic Models* (2020), [arXiv:2006.11239](https://arxiv.org/abs/2006.11239)
- Nichol & Dhariwal — *Improved DDPM* (2021), [arXiv:2102.09672](https://arxiv.org/abs/2102.09672) — cosine schedule
- Song, Meng, Ermon — *Denoising Diffusion Implicit Models* (2021), [arXiv:2010.02502](https://arxiv.org/abs/2010.02502) — DDIM
- Ho & Salimans — *Classifier-Free Diffusion Guidance* (2022), [arXiv:2207.12598](https://arxiv.org/abs/2207.12598)
- Lilian Weng — [*What are Diffusion Models?*](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/) — the clearest derivation of the ELBO